In [3]:
# ============================================================
# STATISTICAL RESULT INTERPRETATION ENGINE
# ============================================================
#
# PURPOSE:
# --------
# This script automatically:
#
# 1. Reads all statistical result files
# 2. Extracts relevant findings
# 3. Removes weak/irrelevant results
# 4. Generates engineering interpretations
# 5. Ranks findings based on importance
# 6. Creates final summary reports
#
# ============================================================
# ANALYSIS COVERED:
#
# - Temperature Analysis
# - Humidity Analysis
# - Delta_T Analysis
#
# FOR:
# - Monthly Analysis
# - Seasonal Analysis
#
# ============================================================

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import os
import pandas as pd
import numpy as np

# ============================================================
# PLACEHOLDER PATHS
# ============================================================
# REPLACE THESE PATHS MANUALLY
# ============================================================

TEMPERATURE_ANALYSIS_PATH = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\MTBF_Analysis\Temperature_Analysis"
HUMIDITY_ANALYSIS_PATH   = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\MTBF_Analysis\Humidity_Analysis"
DELTA_T_ANALYSIS_PATH    = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\MTBF_Analysis\Delta_T_Analysis"

# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIRECTORY = r"ADD_OUTPUT_DIRECTORY_PATH"

os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)

# ============================================================
# THRESHOLDS
# ============================================================

SIGNIFICANCE_LEVEL = 0.05

HIGH_CORRELATION = 0.6
MODERATE_CORRELATION = 0.3

HIGH_CRAMERS_V = 0.5
MODERATE_CRAMERS_V = 0.3

# ============================================================
# STORAGE VARIABLES
# ============================================================

all_findings = []

# ============================================================
# INTERPRET CORRELATION STRENGTH
# ============================================================

def get_correlation_strength(value):

    abs_val = abs(value)

    if abs_val >= HIGH_CORRELATION:
        return "HIGH"

    elif abs_val >= MODERATE_CORRELATION:
        return "MODERATE"

    else:
        return "LOW"

# ============================================================
# INTERPRET CRAMER'S V
# ============================================================

def get_cramers_strength(value):

    if value >= HIGH_CRAMERS_V:
        return "HIGH"

    elif value >= MODERATE_CRAMERS_V:
        return "MODERATE"

    else:
        return "LOW"

# ============================================================
# PARAMETER LIST
# ============================================================

analysis_folders = {

    "Temperature": TEMPERATURE_ANALYSIS_PATH,
    "Humidity": HUMIDITY_ANALYSIS_PATH,
    "Delta_T": DELTA_T_ANALYSIS_PATH

}

# ============================================================
# MAIN LOOP
# ============================================================

for parameter_name, parameter_path in analysis_folders.items():

    print("\n================================================")
    print(f"PROCESSING : {parameter_name}")
    print("================================================")

    # ========================================================
    # MONTHLY + SEASONAL
    # ========================================================

    for analysis_type in ["Monthly_Analysis", "Seasonal_Analysis"]:

        analysis_path = os.path.join(
            parameter_path,
            analysis_type
        )

        print(f"\nAnalyzing : {analysis_type}")

        # ====================================================
        # CORRELATION RESULTS
        # ====================================================

        correlation_folder = os.path.join(
            analysis_path,
            "Correlation_Results"
        )

        if os.path.exists(correlation_folder):

            for file in os.listdir(correlation_folder):

                if file.endswith(".xlsx"):

                    file_path = os.path.join(
                        correlation_folder,
                        file
                    )

                    try:

                        df = pd.read_excel(file_path)

                        for _, row in df.iterrows():

                            # ================================
                            # PEARSON
                            # ================================

                            corr = row.get(
                                'Pearson_Correlation',
                                np.nan
                            )

                            pval = row.get(
                                'Pearson_P_Value',
                                np.nan
                            )

                            if (
                                pd.notna(corr)
                                and
                                pd.notna(pval)
                            ):

                                strength = (
                                    get_correlation_strength(corr)
                                )

                                if (
                                    strength != "LOW"
                                    and
                                    pval < SIGNIFICANCE_LEVEL
                                ):

                                    # ========================
                                    # GROUP NAME
                                    # ========================

                                    possible_groups = [
                                        'Region',
                                        'SLoc'
                                    ]

                                    group_value = "UNKNOWN"

                                    for grp in possible_groups:

                                        if grp in row.index:

                                            group_value = row[grp]
                                            break

                                    # ========================
                                    # INTERPRETATION
                                    # ========================

                                    direction = (
                                        "positive"
                                        if corr > 0
                                        else "negative"
                                    )

                                    interpretation = f"""
{parameter_name} shows a {strength.lower()} {direction} correlation
with MTBF in {group_value}
during {analysis_type.replace('_', ' ')}.
"""

                                    # ========================
                                    # STORE
                                    # ========================

                                    all_findings.append({

                                        "Parameter":
                                            parameter_name,

                                        "Analysis_Type":
                                            analysis_type,

                                        "Test":
                                            "Pearson Correlation",

                                        "Group":
                                            group_value,

                                        "Correlation":
                                            corr,

                                        "P_Value":
                                            pval,

                                        "Relevance":
                                            strength,

                                        "Interpretation":
                                            interpretation.strip()
                                    })

                    except Exception as e:

                        print(f"\nError Reading : {file}")
                        print(e)

        # ====================================================
        # ANOVA RESULTS
        # ====================================================

        anova_folder = os.path.join(
            analysis_path,
            "ANOVA"
        )

        if os.path.exists(anova_folder):

            for file in os.listdir(anova_folder):

                if file.endswith(".xlsx"):

                    file_path = os.path.join(
                        anova_folder,
                        file
                    )

                    try:

                        df = pd.read_excel(file_path)

                        pval = df.loc[0, 'P_Value']

                        if pval < SIGNIFICANCE_LEVEL:

                            interpretation = f"""
Significant group variation detected in
{parameter_name} analysis during
{analysis_type.replace('_', ' ')}.
"""

                            all_findings.append({

                                "Parameter":
                                    parameter_name,

                                "Analysis_Type":
                                    analysis_type,

                                "Test":
                                    "ANOVA",

                                "Group":
                                    file,

                                "Correlation":
                                    np.nan,

                                "P_Value":
                                    pval,

                                "Relevance":
                                    "HIGH",

                                "Interpretation":
                                    interpretation.strip()
                            })

                    except Exception as e:

                        print(f"\nError Reading : {file}")
                        print(e)

        # ====================================================
        # KRUSKAL RESULTS
        # ====================================================

        kruskal_folder = os.path.join(
            analysis_path,
            "Kruskal_Wallis"
        )

        if os.path.exists(kruskal_folder):

            for file in os.listdir(kruskal_folder):

                if file.endswith(".xlsx"):

                    file_path = os.path.join(
                        kruskal_folder,
                        file
                    )

                    try:

                        df = pd.read_excel(file_path)

                        pval = df.loc[0, 'P_Value']

                        if pval < SIGNIFICANCE_LEVEL:

                            interpretation = f"""
Non-parametric significant variation detected in
{parameter_name} analysis during
{analysis_type.replace('_', ' ')}.
"""

                            all_findings.append({

                                "Parameter":
                                    parameter_name,

                                "Analysis_Type":
                                    analysis_type,

                                "Test":
                                    "Kruskal Wallis",

                                "Group":
                                    file,

                                "Correlation":
                                    np.nan,

                                "P_Value":
                                    pval,

                                "Relevance":
                                    "HIGH",

                                "Interpretation":
                                    interpretation.strip()
                            })

                    except Exception as e:

                        print(f"\nError Reading : {file}")
                        print(e)

        # ====================================================
        # TUKEY RESULTS
        # ====================================================

        tukey_folder = os.path.join(
            analysis_path,
            "Tukey_HSD"
        )

        if os.path.exists(tukey_folder):

            for file in os.listdir(tukey_folder):

                if file.endswith(".xlsx"):

                    file_path = os.path.join(
                        tukey_folder,
                        file
                    )

                    try:

                        df = pd.read_excel(file_path)

                        significant_df = df[
                            df['reject'] == True
                        ]

                        for _, row in significant_df.iterrows():

                            interpretation = f"""
Significant difference exists between
{row['group1']} and {row['group2']}
in {parameter_name} analysis.
"""

                            all_findings.append({

                                "Parameter":
                                    parameter_name,

                                "Analysis_Type":
                                    analysis_type,

                                "Test":
                                    "Tukey HSD",

                                "Group":
                                    f"{row['group1']} vs {row['group2']}",

                                "Correlation":
                                    np.nan,

                                "P_Value":
                                    row['p-adj'],

                                "Relevance":
                                    "HIGH",

                                "Interpretation":
                                    interpretation.strip()
                            })

                    except Exception as e:

                        print(f"\nError Reading : {file}")
                        print(e)

        # ====================================================
        # CHI-SQUARE RESULTS
        # ====================================================

        chi_folder = os.path.join(
            analysis_path,
            "Chi_Square"
        )

        if os.path.exists(chi_folder):

            for file in os.listdir(chi_folder):

                if file.endswith(".xlsx"):

                    file_path = os.path.join(
                        chi_folder,
                        file
                    )

                    try:

                        df = pd.read_excel(file_path)

                        pval = df.loc[0, 'P_Value']

                        cramers_v = df.loc[0, 'Cramers_V']

                        relevance = (
                            get_cramers_strength(cramers_v)
                        )

                        if (
                            pval < SIGNIFICANCE_LEVEL
                            and
                            relevance != "LOW"
                        ):

                            interpretation = f"""
{parameter_name} category shows
statistically significant association
with MTBF category during
{analysis_type.replace('_', ' ')}.
"""

                            all_findings.append({

                                "Parameter":
                                    parameter_name,

                                "Analysis_Type":
                                    analysis_type,

                                "Test":
                                    "Chi Square",

                                "Group":
                                    file,

                                "Correlation":
                                    cramers_v,

                                "P_Value":
                                    pval,

                                "Relevance":
                                    relevance,

                                "Interpretation":
                                    interpretation.strip()
                            })

                    except Exception as e:

                        print(f"\nError Reading : {file}")
                        print(e)

# ============================================================
# FINAL FINDINGS DATAFRAME
# ============================================================

findings_df = pd.DataFrame(all_findings)

# ============================================================
# SORT BY RELEVANCE
# ============================================================

relevance_order = {
    "HIGH": 3,
    "MODERATE": 2,
    "LOW": 1
}

findings_df['Relevance_Score'] = (
    findings_df['Relevance']
    .map(relevance_order)
)

findings_df = findings_df.sort_values(

    by=[
        'Relevance_Score',
        'P_Value'
    ],

    ascending=[
        False,
        True
    ]
)

# ============================================================
# SAVE COMPLETE FINDINGS
# ============================================================

complete_output = os.path.join(
    OUTPUT_DIRECTORY,
    "Complete_Statistical_Findings.xlsx"
)

findings_df.to_excel(
    complete_output,
    index=False
)

print("\n================================================")
print("COMPLETE FINDINGS SAVED")
print("================================================")

# ============================================================
# HIGH PRIORITY FINDINGS
# ============================================================

high_priority = findings_df[
    findings_df['Relevance'] == "HIGH"
]

high_output = os.path.join(
    OUTPUT_DIRECTORY,
    "High_Priority_Findings.xlsx"
)

high_priority.to_excel(
    high_output,
    index=False
)

# ============================================================
# MODERATE FINDINGS
# ============================================================

moderate_priority = findings_df[
    findings_df['Relevance'] == "MODERATE"
]

moderate_output = os.path.join(
    OUTPUT_DIRECTORY,
    "Moderate_Priority_Findings.xlsx"
)

moderate_priority.to_excel(
    moderate_output,
    index=False
)

# ============================================================
# SUMMARY REPORT
# ============================================================

summary_lines = []

summary_lines.append(
    "=================================================="
)

summary_lines.append(
    "ENVIRONMENTAL RELIABILITY ANALYSIS SUMMARY"
)

summary_lines.append(
    "=================================================="
)

summary_lines.append(
    f"Total Findings : {len(findings_df)}"
)

summary_lines.append(
    f"High Relevance Findings : {len(high_priority)}"
)

summary_lines.append(
    f"Moderate Relevance Findings : {len(moderate_priority)}"
)

summary_lines.append("\n")

# ============================================================
# TOP 10 FINDINGS
# ============================================================

summary_lines.append(
    "TOP FINDINGS"
)

summary_lines.append(
    "--------------------------------------------------"
)

top_findings = findings_df.head(10)

for idx, row in top_findings.iterrows():

    summary_lines.append(

        f"""
[{row['Relevance']}]

Parameter : {row['Parameter']}
Test      : {row['Test']}
Group     : {row['Group']}
P-Value   : {row['P_Value']}

Interpretation:
{row['Interpretation']}
"""
    )

# ============================================================
# SAVE SUMMARY REPORT
# ============================================================

summary_path = os.path.join(
    OUTPUT_DIRECTORY,
    "Final_Interpretation_Report.txt"
)

with open(summary_path, 'w') as f:

    for line in summary_lines:

        f.write(line + "\n")

# ============================================================
# COMPLETED
# ============================================================

print("\n================================================")
print("STATISTICAL INTERPRETATION ENGINE COMPLETED")
print("================================================")

print("\nGenerated Outputs:")

print("\n1. Complete_Statistical_Findings.xlsx")
print("2. High_Priority_Findings.xlsx")
print("3. Moderate_Priority_Findings.xlsx")
print("4. Final_Interpretation_Report.txt")


PROCESSING : Temperature

Analyzing : Monthly_Analysis

Analyzing : Seasonal_Analysis

PROCESSING : Humidity

Analyzing : Monthly_Analysis

Analyzing : Seasonal_Analysis

PROCESSING : Delta_T

Analyzing : Monthly_Analysis

Analyzing : Seasonal_Analysis

COMPLETE FINDINGS SAVED

STATISTICAL INTERPRETATION ENGINE COMPLETED

Generated Outputs:

1. Complete_Statistical_Findings.xlsx
2. High_Priority_Findings.xlsx
3. Moderate_Priority_Findings.xlsx
4. Final_Interpretation_Report.txt
